In [5]:
import sys
if 'google.colab' in sys.modules:
  !pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 70.9 MB/s eta 0:00:00


In [6]:
# Run this in a new cell
import json
from pydantic import BaseModel, Field, ValidationError
from typing import Optional, List

# Define the strict schema for our insurance claim
class InsuranceClaim(BaseModel):
    patient_name: Optional[str] = Field(default=None)
    policy_number: Optional[str] = Field(default=None)
    hospital_name: Optional[str] = Field(default=None)
    diagnosis: Optional[str] = Field(default=None)
    admission_date: Optional[str] = Field(default=None)
    discharge_date: Optional[str] = Field(default=None)
    total_claim_amount: Optional[str] = Field(default=None)

In [7]:
# ===============================
# Agent-1: Requirements Gathering & Document Similarity Agent
# ===============================

import re
import numpy as np
import faiss
from sklearn.metrics.pairwise import cosine_similarity
from transformers import pipeline
from sentence_transformers import SentenceTransformer


In [8]:
# ===============================
# CONFIGURATION
# ===============================

REQUIRED_FIELDS = [
    "patient_name",
    "policy_number",
    "hospital_name",
    "diagnosis",
    "admission_date",
    "discharge_date",
    "total_claim_amount"
]

SIMILARITY_THRESHOLD = 0.85
EMBEDDING_DIM = 384


In [9]:
# ===============================
# LOAD MODELS
# ===============================

print("Loading LLM model...")
llm = pipeline(
    "text-generation",
    model="google/flan-t5-large",
    max_length=512
)

print("Loading SBERT model...")
sbert = SentenceTransformer("all-MiniLM-L6-v2")

Loading LLM model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [10]:
# ===============================
# FAISS VECTOR DATABASE
# ===============================

faiss_index = faiss.IndexFlatIP(EMBEDDING_DIM)
stored_embeddings = []

In [11]:
import re
# Replace your current extraction function with this
def extract_requirements_json(document_text: str) -> dict:
    prompt = f"""Extract the required insurance fields from the document. Output ONLY a valid JSON object, enclosed within ```json and ```. If a field is missing, use null.

--- Example ---
Document:
Patient: Jane Doe. Admitted to City Care Hospital on 12-May-2023 for Viral Fever. Claim amount: $5000. Policy ID: POL-123.

JSON Output:
```json
{{
  "patient_name": "Jane Doe",
  "policy_number": "POL-123",
  "hospital_name": "City Care Hospital",
  "diagnosis": "Viral Fever",
  "admission_date": "12-May-2023",
  "discharge_date": null,
  "total_claim_amount": "$5000"
}}
```

--- Target Document ---
Document:
{document_text.strip()}

JSON Output:
```json
"""
    # Generate the text
    # Increased max_length to give the LLM more room to generate the JSON
    raw_output = llm(prompt, max_length=512, truncation=True)[0]["generated_text"]
    print(f"DEBUG: LLM Raw Output:\n{raw_output}") # Debug print

    # Use regex to extract the JSON block
    json_match = re.search(r"```json\n(.*?)```", raw_output, re.DOTALL)
    if json_match:
        cleaned_output = json_match.group(1).strip()
    else:
        # Fallback if ```json block not found, try to clean the whole output
        cleaned_output = raw_output.replace('```json', '').replace('```', '').strip()
    print(f"DEBUG: Cleaned Output for JSON.loads:\n{cleaned_output}") # Debug print

    try:
        # 1. Parse the string into a Python Dictionary
        parsed_dict = json.loads(cleaned_output)

        # 2. Push it through the Pydantic validator to ensure it matches our schema exactly
        validated_data = InsuranceClaim(**parsed_dict)

        # Return a clean dictionary
        return validated_data.model_dump()

    except (json.JSONDecodeError, ValidationError) as e:
        print(f"⚠️ Extraction Error: The LLM failed to format the JSON correctly. Error: {e}")
        # Fallback: return an empty claim with all None values so the pipeline doesn't crash
        return InsuranceClaim().model_dump()

In [12]:
# ===============================
# STEP 3: MISSING FIELD DETECTION
# ===============================

# Replace your current detect_missing_fields function with this
def detect_missing_fields_pydantic(validated_dict: dict) -> List[str]:
    # Return a list of keys where the value is None
    return [key for key, value in validated_dict.items() if value is None]

In [13]:
# ===============================
# STEP 4: DOCUMENT EMBEDDING (SIAMESE)
# ===============================

def generate_embedding(text: str):
    embedding = sbert.encode(text, normalize_embeddings=True)
    return embedding.astype("float32")

In [14]:
# ===============================
# STEP 5: DUPLICATE DETECTION (COSINE)
# ===============================

def is_duplicate(embedding, existing_embeddings):
    if len(existing_embeddings) == 0:
        return False, 0.0

    similarities = cosine_similarity(
        [embedding], existing_embeddings
    )[0]

    max_score = np.max(similarities)
    return max_score >= SIMILARITY_THRESHOLD, float(max_score)

In [15]:
# ===============================
# STEP 6: FAISS SEARCH (SCALABLE)
# ===============================

def search_faiss(embedding, k=3):
    if faiss_index.ntotal == 0:
        return [], []

    scores, ids = faiss_index.search(
        np.array([embedding]), k
    )
    return scores[0], ids[0]

In [16]:
# ===============================
# STEP 7: MAIN AGENT-1 PIPELINE
# ===============================

def agent_1_pipeline(document_text: str):
    print("\n--- Agent-1 Processing Started ---")

    # Step 1: LLM-based JSON extraction
    extracted_dict = extract_requirements_json(document_text)
    print("\nExtracted JSON:\n", json.dumps(extracted_dict, indent=2))

    # Step 2: Missing field detection
    missing_fields = detect_missing_fields_pydantic(extracted_dict)
    print("\nMissing Fields:", missing_fields)
    # Step 3: Generate embedding
    embedding = generate_embedding(document_text)

    # Step 4: Duplicate detection
    duplicate, similarity_score = is_duplicate(
        embedding, stored_embeddings
    )

    # Step 5: FAISS search
    scores, ids = search_faiss(embedding)

    # Step 6: Store embedding
    stored_embeddings.append(embedding)
    faiss_index.add(np.array([embedding]))

    print("\nDuplicate:", duplicate)
    print("Similarity Score:", similarity_score)

    return {
        "extracted_requirements": extracted_dict,
        "missing_fields": missing_fields,
        "is_duplicate": duplicate,
        "similarity_score": similarity_score,
        "faiss_matches": {
            "scores": scores.tolist() if len(scores) else [],
            "ids": ids.tolist() if len(ids) else []
        }
    }

In [17]:
# ===============================
# STEP 8: TEST RUN
# ===============================

if __name__ == "__main__":
    # --- IMPORTANT: Ensure all preceding cells (Steps 1-7) have been executed to define 'agent_1_pipeline' and its dependencies. ---

    sample_claim = """
    Patient Name: John Kumar
    Policy Number: ICICI-HEALTH-9981
    Hospital Name: Apollo Hospitals
    Diagnosis: Appendicitis
    Admission Date: 10-Jan-2026
    Total Claim Amount: INR 1,45,000
    """

    output = agent_1_pipeline(sample_claim)

    print("\n--- FINAL AGENT-1 OUTPUT ---")
    print(output)


--- Agent-1 Processing Started ---
DEBUG: LLM Raw Output:
Extract the required insurance fields from the document. Output ONLY a valid JSON object, enclosed within ```json and ```. If a field is missing, use null.

--- Example ---
Document:
Patient: Jane Doe. Admitted to City Care Hospital on 12-May-2023 for Viral Fever. Claim amount: $5000. Policy ID: POL-123.

JSON Output:
```json
{
  "patient_name": "Jane Doe",
  "policy_number": "POL-123",
  "hospital_name": "City Care Hospital",
  "diagnosis": "Viral Fever",
  "admission_date": "12-May-2023",
  "discharge_date": null,
  "total_claim_amount": "$5000"
}
```

--- Target Document ---
Document:
Patient Name: John Kumar
    Policy Number: ICICI-HEALTH-9981
    Hospital Name: Apollo Hospitals
    Diagnosis: Appendicitis
    Admission Date: 10-Jan-2026
    Total Claim Amount: INR 1,45,000

JSON Output:
```json

DEBUG: Cleaned Output for JSON.loads:
{
  "patient_name": "Jane Doe",
  "policy_number": "POL-123",
  "hospital_name": "City Car